# Data Cleaning - VAERSSYMPTOMS.csv

In [1]:
import pandas as pd
import numpy as np
import kagglehub as kh
from kagglehub import KaggleDatasetAdapter

/home/ekmys/PycharmProjects/covid-19-vaccine-adverse-reactions/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Read in + inspect symptoms data

In [3]:
# Download latest csv from Kaggle
path = kh.dataset_download("ayushggarg/covid19-vaccine-adverse-reactions")
print("Path to dataset files:", path)

Path to dataset files: /home/ekmys/.cache/kagglehub/datasets/ayushggarg/covid19-vaccine-adverse-reactions/versions/7


In [4]:
# create symptoms dataframe
path_to_symptoms = path + '/VAERSSYMPTOMS.csv'
df_symptoms = pd.read_csv(path_to_symptoms)
df_symptoms.head()

,VAERS_ID,SYMPTOM1,SYMPTOMVERSION1,SYMPTOM2,SYMPTOMVERSION2,SYMPTOM3,SYMPTOMVERSION3,SYMPTOM4,SYMPTOMVERSION4,SYMPTOM5,SYMPTOMVERSION5
0,902418,Hypoaesthesia,24.0,Injection site hypoaesthesia,24.0,NaN,NaN,NaN,NaN,NaN,NaN
1,902440,Headache,23.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,902446,Erythema,23.1,Feeling hot,23.1,Flushing,23.1,NaN,NaN,NaN,NaN
3,902464,Dizziness,23.1,Electrocardiogram normal,23.1,Hyperhidrosis,23.1,Laboratory test normal,23.1,Presyncope,23.1
4,902465,Dysgeusia,23.1,Oral pruritus,23.1,Paraesthesia,23.1,Paraesthesia oral,23.1,Parosmia,23.1


In [5]:
# inspect shape & info for VAERS symptoms data
df_symptoms.info()
print("Shape of df_symptoms: " , df_symptoms.shape)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1363171 entries, 0 to 1363170
Data columns (total 11 columns):
 #   Column           Non-Null Count    Dtype  
---  ------           --------------    -----  
 0   VAERS_ID         1363171 non-null  int64  
 1   SYMPTOM1         1363171 non-null  object 
 2   SYMPTOMVERSION1  1363171 non-null  float64
 3   SYMPTOM2         1044702 non-null  object 
 4   SYMPTOMVERSION2  1044702 non-null  float64
 5   SYMPTOM3         792563 non-null   object 
 6   SYMPTOMVERSION3  792563 non-null   float64
 7   SYMPTOM4         600471 non-null   object 
 8   SYMPTOMVERSION4  600471 non-null   float64
 9   SYMPTOM5         458415 non-null   object 
 10  SYMPTOMVERSION5  458415 non-null   float64
dtypes: float64(5), int64(1), object(5)
memory usage: 114.4+ MB
Shape of df_symptoms:  (1363171, 11)


## Understand duplicate VAERS_IDs in df_symptoms

In [6]:
#find duplicate rows across specific columns
duplicate_symptoms_df = df_symptoms[df_symptoms.duplicated('VAERS_ID')].sort_values('VAERS_ID')
display(duplicate_symptoms_df)

# Filter to show only duplicated names
duplicate_symptoms_df1 = df_symptoms['VAERS_ID'].value_counts()[df_symptoms['VAERS_ID'].value_counts() > 1]
print("Number of duplicate VAERS IDs in symptoms dataset: ", len(duplicate_symptoms_df1))

,VAERS_ID,SYMPTOM1,SYMPTOMVERSION1,SYMPTOM2,SYMPTOMVERSION2,SYMPTOM3,SYMPTOMVERSION3,SYMPTOM4,SYMPTOMVERSION4,SYMPTOM5,SYMPTOMVERSION5
5,902465,Sensory disturbance,23.1,Tremor,23.1,NaN,NaN,NaN,NaN,NaN,NaN
7,902468,Dyspnoea,24.0,Feeling abnormal,24.0,Flushing,24.0,Presyncope,24.0,NaN,NaN
11,902491,Nausea,23.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
15,902505,Skin warm,23.1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
20,902524,Impaired driving ability,23.1,Limb discomfort,23.1,Nervousness,23.1,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1363138,2776109,Endometritis,27.0,Erythema,27.0,Facial pain,27.0,Gait disturbance,27.0,HLA-B*27 positive,27.0
1363147,2776131,Urticaria,27.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1363152,2776214,Electrocardiogram,27.0,Immunisation reaction,27.0,Laboratory test,27.0,Myopericarditis,27.0,Palpitations,27.0
1363154,2776226,Myopericarditis,27.0,Stress cardiomyopathy,27.0,NaN,NaN,NaN,NaN,NaN,NaN


Number of duplicate VAERS IDs in symptoms dataset:  241896


In [7]:
# Count occurrences of each VAERS ID in symptoms dataset
ID_counts_symptoms = df_symptoms['VAERS_ID'].value_counts()
print("Count of each VAERS ID in symptoms dataset:")
print(ID_counts_symptoms)

Count of each VAERS ID in symptoms dataset:
VAERS_ID
2685738    36
2500895    33
2697023    32
2050891    32
2062218    29
           ..
1415129     1
1415130     1
1415133     1
1415134     1
1415109     1
Name: count, Length: 1012889, dtype: int64


**Inspection Summary**
- VAERSSYMPTOMS.csv contains patient symptoms per adverse event as coded in the "MedDRA dictionary" (User Guide, p. 8).
- Shape: df_symptoms contains 11 columns and 1,363,171 rows
- Most columns have missing data / null values except for the first three: VAERS_ID, SYMPTOM1, and SYMPTOMVERSION1, but this seems intuitive / reasonable. Any adverse event reported required the patient to have at least one symptom, but not all patients had more than one symptom. So it makes sense we're seeing null values in the subsequent columns that describe additional symptoms.
- NOTE - There are 241896 duplicate VAERS IDs in the symptoms dataset. This is likely because the VAERS event forms only included up to 5 symptom slots, so if patients were experiencing more than 5 symptoms they needed to fill out a second form. "Each row in the .csv will contain up to 5 MedDRA terms per VAERS ID; thus, there could be multiple rows per VAERS ID" (User Guide, p. 8). Will need to resolve this in data cleaning.

## Reshape symptoms dataframe to something more manageable

In [8]:
# convert SYMPTOM / SYMPTOMVERSION pairs to dictionary item in series
VAERS_IDs_dict = df_symptoms[['VAERS_ID']].apply(lambda row: row.to_dict(), axis=1)
symptom1_dict = df_symptoms[['SYMPTOM1','SYMPTOMVERSION1']].apply(lambda row: row.to_dict(), axis=1)
symptom2_dict = df_symptoms[['SYMPTOM2','SYMPTOMVERSION2']].apply(lambda row: row.to_dict(), axis=1)
symptom3_dict = df_symptoms[['SYMPTOM3','SYMPTOMVERSION3']].apply(lambda row: row.to_dict(), axis=1)
symptom4_dict = df_symptoms[['SYMPTOM4','SYMPTOMVERSION4']].apply(lambda row: row.to_dict(), axis=1)
symptom5_dict = df_symptoms[['SYMPTOM5','SYMPTOMVERSION5']].apply(lambda row: row.to_dict(), axis=1)

In [9]:
# verify symptom dictionary creation (uncomment to run)
# symptom1_dict
# symptom2_dict
# symptom3_dict
# symptom4_dict
# symptom5_dict
# VAERS_IDs_dict

In [10]:
# create subset dataset of VAERS_IDs
df_VAERS_IDs = df_symptoms['VAERS_ID'].to_frame()
df_VAERS_IDs.head()

,VAERS_ID
0,902418
1,902440
2,902446
3,902464
4,902465


In [11]:
# convert symptom1_dict through symptom5_dict to dataframe
symptom1_df = symptom1_dict.to_frame()
symptom2_df = symptom2_dict.to_frame()
symptom3_df = symptom3_dict.to_frame()
symptom4_df = symptom4_dict.to_frame()
symptom5_df = symptom5_dict.to_frame()

In [12]:
# verify conversion (uncomment to run)
# print(type(symptom1_df))

In [13]:
# add column with VAERS_ID values to new dataframes
symptom1_df['VAERS_ID'] = df_VAERS_IDs
symptom2_df['VAERS_ID'] = df_VAERS_IDs
symptom3_df['VAERS_ID'] = df_VAERS_IDs
symptom4_df['VAERS_ID'] = df_VAERS_IDs
symptom5_df['VAERS_ID'] = df_VAERS_IDs

In [14]:
# verify column addition (uncomment to run)
symptom1_df.head()

,0,VAERS_ID
0,"{'SYMPTOM1': 'Hypoaesthesia', 'SYMPTOMVERSION1...",902418
1,"{'SYMPTOM1': 'Headache', 'SYMPTOMVERSION1': 23.1}",902440
2,"{'SYMPTOM1': 'Erythema', 'SYMPTOMVERSION1': 23.1}",902446
3,"{'SYMPTOM1': 'Dizziness', 'SYMPTOMVERSION1': 2...",902464
4,"{'SYMPTOM1': 'Dysgeusia', 'SYMPTOMVERSION1': 2...",902465


In [15]:
# add columns for 'symptom' and 'symptom version' to each frame
symptom1_df['symptom'] = symptom1_df[0].apply(lambda row: row['SYMPTOM1'])
symptom1_df['symptom version'] = symptom1_df[0].apply(lambda row: row['SYMPTOMVERSION1'])

symptom2_df['symptom'] = symptom2_df[0].apply(lambda row: row['SYMPTOM2'])
symptom2_df['symptom version'] = symptom2_df[0].apply(lambda row: row['SYMPTOMVERSION2'])

symptom3_df['symptom'] = symptom3_df[0].apply(lambda row: row['SYMPTOM3'])
symptom3_df['symptom version'] = symptom3_df[0].apply(lambda row: row['SYMPTOMVERSION3'])

symptom4_df['symptom'] = symptom4_df[0].apply(lambda row: row['SYMPTOM4'])
symptom4_df['symptom version'] = symptom4_df[0].apply(lambda row: row['SYMPTOMVERSION4'])

symptom5_df['symptom'] = symptom5_df[0].apply(lambda row: row['SYMPTOM5'])
symptom5_df['symptom version'] = symptom5_df[0].apply(lambda row: row['SYMPTOMVERSION5'])

In [16]:
# verify column additions (uncomment to run)
# symptom1_df.head()

In [17]:
# concatenate all dataframes into one
symptom_dataframes = [symptom1_df, symptom2_df, symptom3_df, symptom4_df, symptom5_df]
df_symptoms_reshaped = pd.concat(symptom_dataframes)

# Code example from: https://pandas.pydata.org/docs/user_guide/merging.html

In [18]:
# verify dataframe reshaped correctly
print(len(symptom1_df))
print(len(df_symptoms_reshaped))

1363171
6815855


In [19]:
df_symptoms_reshaped.head()

,0,VAERS_ID,symptom,symptom version
0,"{'SYMPTOM1': 'Hypoaesthesia', 'SYMPTOMVERSION1...",902418,Hypoaesthesia,24.0
1,"{'SYMPTOM1': 'Headache', 'SYMPTOMVERSION1': 23.1}",902440,Headache,23.1
2,"{'SYMPTOM1': 'Erythema', 'SYMPTOMVERSION1': 23.1}",902446,Erythema,23.1
3,"{'SYMPTOM1': 'Dizziness', 'SYMPTOMVERSION1': 2...",902464,Dizziness,23.1
4,"{'SYMPTOM1': 'Dysgeusia', 'SYMPTOMVERSION1': 2...",902465,Dysgeusia,23.1


In [20]:
# reorder columns
df_symptoms_reshaped = df_symptoms_reshaped.iloc[:, [1, 2, 3, 0]]  #
df_symptoms_reshaped.head()

# Code from: https://www.geeksforgeeks.org/python/change-the-order-of-a-pandas-dataframe-columns-in-python/

,VAERS_ID,symptom,symptom version,0
0,902418,Hypoaesthesia,24.0,"{'SYMPTOM1': 'Hypoaesthesia', 'SYMPTOMVERSION1..."
1,902440,Headache,23.1,"{'SYMPTOM1': 'Headache', 'SYMPTOMVERSION1': 23.1}"
2,902446,Erythema,23.1,"{'SYMPTOM1': 'Erythema', 'SYMPTOMVERSION1': 23.1}"
3,902464,Dizziness,23.1,"{'SYMPTOM1': 'Dizziness', 'SYMPTOMVERSION1': 2..."
4,902465,Dysgeusia,23.1,"{'SYMPTOM1': 'Dysgeusia', 'SYMPTOMVERSION1': 2..."


In [21]:
# rename dictionary column for reference
df_symptoms_reshaped = df_symptoms_reshaped.set_axis(['VAERS_ID', 'symptom', 'symptom_version', 'symptom_dict'], axis=1)

In [22]:
df_symptoms_reshaped.head()

,VAERS_ID,symptom,symptom_version,symptom_dict
0,902418,Hypoaesthesia,24.0,"{'SYMPTOM1': 'Hypoaesthesia', 'SYMPTOMVERSION1..."
1,902440,Headache,23.1,"{'SYMPTOM1': 'Headache', 'SYMPTOMVERSION1': 23.1}"
2,902446,Erythema,23.1,"{'SYMPTOM1': 'Erythema', 'SYMPTOMVERSION1': 23.1}"
3,902464,Dizziness,23.1,"{'SYMPTOM1': 'Dizziness', 'SYMPTOMVERSION1': 2..."
4,902465,Dysgeusia,23.1,"{'SYMPTOM1': 'Dysgeusia', 'SYMPTOMVERSION1': 2..."


## Cleaning (Dropping NA Values)

In [23]:
# count NA values
df_symptoms_reshaped.isna().sum()

VAERS_ID                 0
symptom            2556533
symptom_version    2556533
symptom_dict             0
dtype: int64

In [24]:
# drop NA values
df_symptoms_reshaped = df_symptoms_reshaped.dropna()

In [25]:
# verify dropped NA values
df_symptoms_reshaped.isna().sum()

VAERS_ID           0
symptom            0
symptom_version    0
symptom_dict       0
dtype: int64

In [26]:
# count unique values
value_counts = df_symptoms_reshaped['VAERS_ID'].value_counts()[df_symptoms_reshaped['VAERS_ID'].value_counts() > 1]
print("\nDuplicated VAERS ID's in transformed symptoms dataset and their counts:")
print(value_counts)
print("Number of duplicate VAERS IDs in transformed symptoms dataset: ", len(value_counts))

# Code from: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.duplicated.html


Duplicated VAERS ID's in transformed symptoms dataset and their counts:
VAERS_ID
2685738    176
2500895    164
2697023    157
2050891    157
2062218    144
          ... 
1522283      2
2130464      2
2130451      2
2130455      2
2130437      2
Name: count, Length: 774970, dtype: int64
Number of duplicate VAERS IDs in transformed symptoms dataset:  774970


Now that we've reshaped the dataframe, there should be significantly more duplicate VAERS IDs for patients who had more than one symptom.

There are now 4,259,322 rows in df_symptoms_reshaped, as opposed to the 1,363,171 rows in df_symptoms before reshaping and dropping N/A values.

In [27]:
# inspect shape & info for transformed symptoms data
df_symptoms_reshaped.info()
print("Shape of df_symptoms: " , df_symptoms_reshaped.shape)

<class 'pandas.core.frame.DataFrame'>
Index: 4259322 entries, 0 to 1363157
Data columns (total 4 columns):
 #   Column           Dtype  
---  ------           -----  
 0   VAERS_ID         int64  
 1   symptom          object 
 2   symptom_version  float64
 3   symptom_dict     object 
dtypes: float64(1), int64(1), object(2)
memory usage: 162.5+ MB
Shape of df_symptoms:  (4259322, 4)


# Remaining Cleaning

In [28]:
# remove whitespace from string columns
df_symptoms_reshaped['symptom'] = df_symptoms_reshaped['symptom'].str.strip()

In [29]:
# save transformed symptoms dataframe to csv
path_to_save = path + '/VAERSSYMPTOMS_cleaned.csv'
path_to_save

# save transformed symptoms dataframe to csv
df_symptoms_reshaped.to_csv(path_to_save)